# 06 · EDA de perfilado de zonas para clustering (Issue #25)

**Responsable:** Integrante 3 (Clustering + Dashboard) · **Fase CRISP-ML:** 3 · **Alimenta a:** #26 (features + línea base z-score), #27 (K-Means formal).

Análisis exploratorio sobre el **dataset analítico unificado** (`data/03_primary/dataset_analitico.parquet`),
para caracterizar cómo se compone el delito en cada localidad (no solo cuánto, sino **de qué tipo**) y
adelantar una hipótesis visual de en cuántos perfiles distintos podrían agruparse las 20 localidades.

> **Alcance:** este notebook es EDA, no modelado. El K-Means formal (features estandarizadas con NUSE/IPM,
> elbow + silhouette cuantitativos, `k` final justificado) es la Issue #27; aquí solo se explora visualmente.

**Secciones**
1. Perfiles delictivos por localidad (proporción de cada tipo de delito)
2. Tendencia por año por localidad-tipo (¿sube o baja?)
3. Hipótesis preliminar de cuántos perfiles existen (dendrograma)


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress, zscore
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Paleta del proyecto (consistente con 01_EDA_exploracion_datos.ipynb, el dashboard y la app movil)
COLORES = {
    "bajo":   "#17D05B",
    "medio":  "#F5A623",
    "alto":   "#E05252",
    "neutro": "#C89B3C",
    "fondo":  "#0D1117",
}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})
sns.set_style("whitegrid")

DF = pd.read_parquet("../data/03_primary/dataset_analitico.parquet")
DF["localidad_nombre"] = DF["localidad_nombre"].astype("string").astype(object)
DF["tipo_delito_nombre"] = DF["tipo_delito_nombre"].astype("string").astype(object)
print("Dataset:", DF.shape, "| columnas:", list(DF.columns))
print("Localidades:", DF["localidad_nombre"].nunique(), "| tipos de delito:", DF["tipo_delito_nombre"].nunique(),
      "| anios:", sorted(DF["anio"].unique()))
assert DF.shape == (1760, 11), "forma inesperada del dataset analitico"
assert DF["localidad_nombre"].nunique() == 20 and DF["tipo_delito_nombre"].nunique() == 11
DF.head()


Dataset: (1760, 11) | columnas: ['cod_localidad', 'localidad_nombre', 'anio', 'tipo_delito', 'tipo_delito_nombre', 'conteo_siedco', 'conteo_nuse', 'poblacion', 'ipm_nbi', 'split', 'riesgo_alto']
Localidades: 20 | tipos de delito: 11 | anios: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,cod_localidad,localidad_nombre,anio,tipo_delito,tipo_delito_nombre,conteo_siedco,conteo_nuse,poblacion,ipm_nbi,split,riesgo_alto
0,01,USAQUEN,2018,DS,Delitos Sexuales,263,223460,534118,2.424007,train,0
1,01,USAQUEN,2018,H,Homicidios,36,223460,534118,2.424007,train,0
2,01,USAQUEN,2018,HA,Hurto Automotores,158,223460,534118,2.424007,train,0
3,01,USAQUEN,2018,HB,Hurto Bicicletas,789,223460,534118,2.424007,train,1
4,01,USAQUEN,2018,HC,Hurto Comercio,1295,223460,534118,2.424007,train,1
